# 30 Diffbot descarga textos

Al ejecutar la primera celda, se pide la autorización para acceder a Gdrive. A veces falla en el primer intento, volver a intentarlo.

In [ ]:
import gspread
from google.oauth2.service_account import Credentials

import pandas as pd

url_sin_detenidos = 'https://docs.google.com/spreadsheets/d/1xKAdnEC8HP4b5MJ-HrOGKqa7cXE-ue336RsWkIAhnYg/edit?gid=762130750#gid=762130750'

try:
    from google.colab import auth
    import google.auth

    # Running in Colab
    auth.authenticate_user()
    creds, _ = google.auth.default()
    client = gspread.authorize(creds)

except ImportError:
    from google.oauth2.service_account import Credentials

    # Running locally or outside Colab
    SCOPES = [
        "https://www.googleapis.com/auth/spreadsheets",
        "https://www.googleapis.com/auth/drive",
    ]
    creds = Credentials.from_service_account_file(
        "../secrets/credentials.json", scopes=SCOPES
    )
    client = gspread.authorize(creds)


spreadsheet = client.open_by_url(url_sin_detenidos)

### Dataframe con artículos (sin el texto) registrados por el RSS

In [ ]:
# ws = spreadsheet.get_worksheet(0)
ws = spreadsheet.worksheet('INBOX')

# 1. Get raw cell values as a 2D list
raw_values = ws.get_all_values()

# 2. Define your own custom header names
headers = ['Fecha_deteccion', 'Medio', 'Titulo', 'Link', 'Keyword_detectada',
        'Fuente', 'Revisado', 'Validado', 'Observaciones', 'Estado_IA',
        'Palabras_detectadas', 'Puntaje', 'Puntaje_solo_titulo', 'otro_2']

# 3. Map values to records (skipping row 0 if row 0 has the old headers)
all_records = gspread.utils.to_records(headers, raw_values[1:])

# Example output
# for row in all_records[:2]:  # Print first 2 rows
#     print(row)

# {'Fecha_deteccion': 'desde aca ejecuté el nuevo código', 'Medio': '', 'Titulo': '', 'Link': '', 'Keyword_detectada': '', 'Fuente': '', 'Revisado': 'FALSE', 'Validado': '', 'Observaciones': '', 'Estado_IA': '', 'Palabras_detectadas': '', 'Puntaje': '', 'Puntaje_solo_titulo': '', 'otro_2': ''}
# {'Fecha_deteccion': '4/8/2026', 'Medio': 'Google News', 'Titulo': 'Facundo Moyano fue demorado en Belgrano: pelea de pareja y estupefacientes - andigital.com.ar', 'Link': 'https://news.google.com/rss/articles/CBMisAFBVV95cUxOMW5QRFdDeFRrZkVKbGFqTHljUnVMQkVSV1pRazQ4eXo3ajAwR1kzMDRWN2JaY01HOEdpeEp0TmFvQS04Z2VEMDdXQ3AyT29lemwzRHhTUURndUhrRDAtWU0tT3R1N1pmejd6c2hfQ0I5UnV2emlQdmthczZUX0FIbkRpY2NEY0Z0c0k4ZkJUNFhWMWZxdHF3ZTNndVR2ZDRkdU5MSW1PYmItU1kwd0lwUw?oc=5', 'Keyword_detectada': 'demorado', 'Fuente': 'https://news.google.com/rss/search?q=demorado%20CABA&hl=es-419&gl=AR&ceid=AR:es-419', 'Revisado': 'FALSE', 'Validado': 'PENDIENTE', 'Observaciones': '', 'Estado_IA': 'PROCESADO', 'Palabras_detectadas': 'POLICIA: policia, efectivo, uniformado, comisaria, policial, GNA, DIR | VICTIMA: demorado, victima | VIOLENCIA_POLICIAL: operativo | VIOLENCIA_GENERAL: incidente, golpe, violencia | CABA: ciudad de buenos aires, Belgrano, hospital pirovano', 'Puntaje': '12', 'Puntaje_solo_titulo': '', 'otro_2': ''}

# Convert records directly to DataFrame
df_sd = pd.DataFrame(all_records)

# Remove invalid rows: no Title
col = df_sd.columns[2]

# Filter out empty/whitespace strings and NaNs
df_sd = df_sd[df_sd[col].astype(str).str.strip().ne("") & df_sd[col].notna()]

# Reset index, keep the old index for reference, matching the row number in the gdrive sheet.
df_sd.reset_index(names="gdrive_index", inplace=True)

# Remove column "Medio", it's always equal to "Google News"
df_sd.drop(columns=["Medio"], inplace=True)

# Remove column "Fuente", it shows the search source URL, not the url for the article
df_sd.drop(columns=["Fuente"], inplace=True)

# Últimos artículos
df_sd.iloc[-5:]

El RSS devuelve artículos de cualquier fecha, no las últimas noticias. Acceder a algún artículo para verificar que la fecha es de años anteriores.

In [ ]:
print(df_sd.iloc[402]['Link'])

## Diffbot

In [ ]:
import json
import os
import time
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()  # Automatically finds .env file
api_key = os.getenv('DIFFBOT_API_KEY')

base_dir = Path.cwd()
# csv_path = base_dir / ".." / "data" / "Monitoreo noticias with articles.csv"
# output_csv_path = csv_path

# df = pd.read_csv(csv_path)
# print(f"Number of rows in DataFrame: {len(df)}")

n_i = 397
n_f = 399

print(f"Processing rows {n_i} to {n_f} from DataFrame...")
print("Waiting 1 minute between requests to avoid hitting the rate limit...")

if "diffbot_response" not in df_sd.columns:
    df_sd["diffbot_response"] = pd.NA

url = f"https://api.diffbot.com/v3/article?token={api_key}"
headers = {}

subset = df_sd.iloc[n_i:n_f]


# Filter only rows that do not have a diffbot response already.
def needs_diffbot_response(value):
    if pd.isna(value):
        return True
    if not isinstance(value, str):
        return True
    text = value.strip()
    if not text:
        return True
    try:
        parsed = json.loads(text)
    except (ValueError, TypeError):
        return True
    return not (isinstance(parsed, dict) and "request" in parsed)

missing_response = subset["diffbot_response"].apply(needs_diffbot_response)

# for index, row in subset[missing_response].iterrows():
for index, row in subset.iterrows():
    print(f"Processing row {index} (ID = {row.get('ID')})...", end="")
    link = row.get("Link")
    if pd.isna(link) or not str(link).strip():
        print(f" skipping, missing link")
        continue

    try:
        params = {
            "url": link
        }
        response = requests.request("GET", url, params=params, headers=headers)
        print(f" status={response.status_code}")
        response_json = response.json()
        df_sd.at[index, "diffbot_response"] = json.dumps(response_json)
        # df_sd.to_csv(output_csv_path, index=False)
    except requests.RequestException as exc:
        print(f" request failed: {exc}")

    time.sleep(65)  # Sleep for 65 seconds to avoid hitting the rate limit


# print(f"Saved dataframe to {output_csv_path}")


In [ ]:
df_sd.iloc[395:400]['diffbot_response']

In [ ]:
# obj = json.loads((df_sd.iloc[399]['diffbot_response']) or '{}')
# json_obj = (obj.get('objects') or [{}])[0]
# # print(json_obj.get('resolvedPageUrl'))
# print(json_obj.get('text'))

# # Save to txt
# with open("output_399.txt", "w", encoding="utf-8") as f:
#     f.write(json_obj.get('text') or '')

import re

if "archivo" not in df_sd.columns:
    df_sd["archivo"] = pd.NA

# Find the greatest existing number in "archivo" values like output_0001.txt
pattern = re.compile(r"^output_(\d+)\.txt$")
existing_numbers = [
    int(m.group(1))
    for value in df_sd["archivo"].dropna()
    if (m := pattern.match(str(value).strip()))
]
next_number = (max(existing_numbers) + 1) if existing_numbers else 1

for i in range(n_i, n_f):
    obj = json.loads((df_sd.iloc[i]['diffbot_response']) or '{}')
    json_obj = (obj.get('objects') or [{}])[0]
    text = json_obj.get('text') or ''
    print(f"Row {i}: {len(text)} chars")

    filename = f"output_{next_number:04d}.txt"
    with open(filename, "w", encoding="utf-8") as f:
        f.write(text)

    df_sd.at[df_sd.index[i], "archivo"] = filename
    next_number += 1

## Observaciones

- Cuando se busca 'gatillo fácil', siempre aparecen artículos de https://www.laizquierdadiario.cl aunque en los artículos no aparezca 'gatillo' ni 'fácil'.
- Incluir las palabras con y sin tilde dan resultados diferentes, por ejemplo: represión y represion. Aunque la palabra esté bien escrita en el artículo (un artículo que contiene la palabra represión), algunos artículos no aparecen en la búsqueda por 'represión' y sí aparecen cuando la búsqueda es 'represion'. Sin embargo, hacerlo con todas las palabras que tienen tilde agrega muchas búsquedas, por lo cual no se hace con todas. Es cuestión de experimentar si conviene agregarla o no.
- https://www.laizquierdadiario.cl muestra artículos de Argentina.
- revista crisis, instagram, facebook: ¿excluir?
- country='AR' , Google ignores this